# 부록A. 신규 고객 RFM 변수 만들기

- **분석 배경**
  - 로지스틱 회귀와 머신러닝 실습에서는 고객별 변수가 이미 정리된 분석 테이블(`repurchase.csv`)을 사용했습니다.
  - 실무에서는 이런 테이블이 주어지지 않으므로 고객 마스터와 거래 내역에서 직접 만들어야 합니다.
- **분석 대상**
  - 2025년 7 ~ 12월에 첫 구매를 한 신규 고객 중 6차월 말(기준일)까지 탈퇴하지 않은 고객입니다.
- **분석 질문**
  - 거래 내역으로 고객별 RFM 변수와 행동 변수를 만들고, 재구매 분석 테이블을 직접 완성할 수 있는가?
- **사용 데이터**
  - `new_customers.csv`(고객 마스터 10,000명)
  - `transactions.csv`(거래 내역 53,448행)
  - `repurchase.csv`(검증용 분석 테이블 9,173명)
- **분석 단위**: 신규 고객

**학습 목표**

1. 기준일을 정하고 기준일 이전 거래만 남겨 정보 누수를 막을 수 있다.
2. 거래 내역을 고객 단위로 집계해 RFM 변수(최근성·구매 횟수·구매 금액)를 만들 수 있다.
3. 활동 월 수, 신선식품·할인 결제 비중, 채널 변수 같은 행동 변수를 만들 수 있다.
4. 목표변수(7차월 재구매 여부)를 만들어 분석 테이블을 완성하고 검증할 수 있다.
5. RFM 점수로 고객 등급을 나누고 등급별 재구매율을 비교할 수 있다.

## 0. 준비

### 라이브러리 불러오기

In [ ]:
# 필요한 모듈을 임포트합니다.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import hds

### 그래프 한글 폰트 설정

구글 폰트(고운바탕)를 내려받아 그래프 한글 폰트로 설정합니다.<br>
음수 부호(−)가 네모로 깨지지 않도록 하는 설정도 함께 처리합니다.

In [ ]:
# 구글 폰트(고운바탕)를 내려받아 그래프 한글 폰트로 설정합니다.
# 처음 실행할 때만 인터넷 연결이 필요하고, 이후에는 내려받은 파일을 다시 사용합니다.
hds.plot.set_font(font_name='Gowun Batang')

# 그래프 제목을 굵게 표시합니다.
plt.rc(group='axes', titleweight='bold')

### 데이터 불러오기

In [ ]:
# 신규 고객 마스터 csv 파일을 읽고 customers에 할당합니다.
customers = pd.read_csv(filepath_or_buffer='../data/new_customers.csv', encoding='utf-8-sig')

# customers의 처음 5행과 마지막 5행을 확인합니다.
# [참고] 데이터프레임 이름만 실행하면 처음 5행과 마지막 5행, 행 개수와 열 개수를 함께 보여줍니다.
customers

In [ ]:
# customers의 정보를 확인합니다.
customers.info()

In [ ]:
# 거래 내역 csv 파일을 읽고 transactions에 할당합니다.
transactions = pd.read_csv(filepath_or_buffer='../data/transactions.csv', encoding='utf-8-sig')

# transactions의 처음 5행과 마지막 5행을 확인합니다.
transactions

In [ ]:
# transactions의 정보를 확인합니다.
transactions.info()

In [ ]:
# 재구매 분석용 csv 파일을 읽고 repurchase에 할당합니다.
repurchase = pd.read_csv(filepath_or_buffer='../data/repurchase.csv', encoding='utf-8-sig')

# repurchase의 처음 5행과 마지막 5행을 확인합니다.
repurchase

In [ ]:
# repurchase의 정보를 확인합니다.
repurchase.info()

**해석**
- `repurchase.csv`는 6절에서 직접 만든 테이블과 비교하는 검증용으로만 사용합니다.
- 날짜 열(`cutoff_date`, `txn_date` 등)이 문자열로 읽혔으므로 날짜시간형으로 변환해야 합니다.

## 1. 분석 설계

**분석 설계**

```
M1 ─ M2 ─ M3 ─ M4 ─ M5 ─ M6 │ M7
└──── 입력변수 관찰 ────┘  기준일  └ 목표변수: 재구매 여부
                       (M6 말일)
```

- M1은 고객의 첫 구매월(2025-07 ~ 2025-12)입니다.
  - 고객마다 M1이 다르므로 기준일도 고객마다 다릅니다.
- 입력변수와 목표변수를 만드는 기준은 다음과 같습니다.
  - **입력변수**는 기준일까지의 거래로만 만들어, 데이터 누수를 방지합니다.
  - **목표변수**는 기준일 다음 달(M7)의 재구매 여부입니다.
- 기준일 **전에 탈퇴한 고객**은 M7에 재구매할 수 없으므로 분석에서 제외합니다.
  - 이미 탈퇴한 고객을 분석 대상에 포함하면 '재구매하지 않음'이 탈퇴 때문인지 행동 때문인지 구분할 수 없습니다.
- RFM은 최근성(Recency), 구매 횟수(Frequency), 구매 금액(Monetary)의 머리글자입니다.
  - 고객의 구매 활동을 요약하는 대표적인 변수이며, CRM 마케팅 분야에서 대표적인 RFM 분석에 사용됩니다.

## 2. 기준일 결합과 정보 누수 방지

**질문**: 날짜 열을 날짜시간형으로 바꾸고, 기준일 전에 탈퇴한 고객을 제외하면 몇 명이 남는가?

In [ ]:
# customers의 기준일을 날짜시간형으로 변환합니다.
customers['cutoff_date'] = pd.to_datetime(arg=customers['cutoff_date'])

# customers의 첫 구매일을 날짜시간형으로 변환합니다.
customers['first_purchase_date'] = pd.to_datetime(arg=customers['first_purchase_date'])

# customers의 탈퇴일을 날짜시간형으로 변환합니다.
# [참고] 유지 고객의 탈퇴일은 결측값(NaT)으로 채워집니다.
customers['withdraw_date'] = pd.to_datetime(arg=customers['withdraw_date'])

# 기준일 이전 탈퇴 여부를 withdrawn에 할당합니다.
# [참고] 탈퇴일이 결측값(NaT)이면 비교 결과가 False입니다.
withdrawn = customers['withdraw_date'].le(other=customers['cutoff_date'])

# customers에서 탈퇴 고객을 제외하고 active_customers에 할당합니다.
active_customers = customers.loc[~withdrawn, :]

# transactions의 구매일을 날짜시간형으로 변환합니다.
transactions['txn_date'] = pd.to_datetime(arg=transactions['txn_date'])

# 전체 고객 수, 탈퇴 고객 수, 분석 대상 수를 확인합니다.
print('전체 신규 고객:', customers.shape[0])
print('기준일 전 탈퇴:', withdrawn.sum())
print('분석 대상 행 개수:', active_customers.shape[0])
print('repurchase 행 개수:', repurchase.shape[0])

In [ ]:
# active_customers의 customer_id 고유값을 active_customers_id에 할당합니다.
active_customers_id = active_customers['customer_id'].unique()

# repurchase의 customer_id 고유값을 repurchase_id에 할당합니다.
repurchase_id = repurchase['customer_id'].unique()

# active_customers_id와 repurchase_id가 서로 일치하는지 확인합니다.
active_customers_id.equals(other=repurchase_id)

**해석**
- 분석 대상 수가 `repurchase.csv`의 행 수와 같습니다.
- 분석 대상 고객 ID와 `repurchase.csv`의 고객 ID도 서로 일치합니다.

**질문**: 거래 내역에 고객별 첫 구매일과 기준일을 붙이면, 첫 구매일 이전 거래와 기준일 이후 거래는 몇 건인가?

In [ ]:
# 분석 대상 고객의 거래 내역에 고객별 첫 구매일과 기준일을 결합하고 txns_cutoff에 할당합니다.
txns_cutoff = transactions.merge(right=active_customers[['customer_id', 'first_purchase_date', 'cutoff_date']], on='customer_id')

# 첫 구매일(M1) 이전 거래 여부를 before_first에 할당합니다.
before_first = txns_cutoff['txn_date'].lt(other=txns_cutoff['first_purchase_date'])

# 기준일 이후 거래 여부를 after_cutoff에 할당합니다.
after_cutoff = txns_cutoff['txn_date'].gt(other=txns_cutoff['cutoff_date'])

# 첫 구매일부터 기준일까지의 거래만 남기고 txns_before에 할당합니다.
txns_before = txns_cutoff.loc[~before_first & ~after_cutoff, :].copy()

# 분석 대상 고객의 거래 건수, 첫 구매일 이전 거래 건수, 기준일 이후 거래 건수를 확인합니다.
print('분석 대상 고객의 거래:', txns_cutoff.shape[0])
print('첫 구매일 이전 거래:', before_first.sum())
print('기준일 이후 거래:', after_cutoff.sum())
print('기준일 이전 거래:', txns_before.shape[0])

**해석**
- 첫 구매일 이전 거래는 0건입니다.
  - 신규 고객은 M1이 첫 구매월이므로 그 이전 거래가 없습니다.
  - 실무 거래 테이블에는 관찰 기간 밖의 거래가 섞여 있을 수 있으므로, 관찰 시작일(M1)과 기준일 양쪽으로 걸러야 합니다.
- 거래 내역에는 목표변수 산정 기간(M7)의 거래도 들어 있습니다.
  - 이 거래로 입력변수를 계산하면 **예측하려는 미래가 입력변수에 섞여 들어가는 정보 누수**가 생깁니다.
  - 입력변수는 반드시 `txns_before`(기준일 이전 거래)로만 만들어야 합니다.
- 기준일 이후 거래는 5절에서 목표변수를 만들 때 사용합니다.

## 3. RFM 변수 만들기

**질문**: 기준일 이전 거래로 고객별 `frequency`(구매 횟수), `monetary`(구매 금액), `recency_days`(마지막 구매 후 경과일)을 계산합니다.

In [ ]:
# 고객별 구매 횟수, 구매 금액 합계, 마지막 구매일, 기준일을 계산하고 rfm에 할당합니다.
rfm = txns_before.groupby(by='customer_id').agg(frequency=('txn_id', 'count'),
                                                monetary=('amount', 'sum'),
                                                last_date=('txn_date', 'max'),
                                                cutoff_date=('cutoff_date', 'first'))

# 기준일에서 마지막 구매일을 뺀 일수를 recency_days로 추가합니다.
rfm['recency_days'] = (rfm['cutoff_date'] - rfm['last_date']).dt.days

# rfm의 처음 5행을 확인합니다.
rfm.head()

**해석**
- 고객별 RFM 변수는 다음과 같습니다.

  | 변수 | 뜻 | 해석 |
  |---|---|---|
  | `frequency` | 기준일까지 구매 횟수 | 클수록 여러 번 구매한 고객 |
  | `monetary` | 기준일까지 구매 금액 합계(원) | 클수록 많은 금액을 쓴 고객 |
  | `recency_days` | 기준일 − 마지막 구매일 | 클수록 오랫동안 구매하지 않은 고객 |

  [참고] 최근성은 오늘 날짜가 아니라 **기준일**에서 뺍니다. 오늘 날짜로 계산하면 분석하는 날마다 값이 바뀌고, 기준일 이후의 정보가 섞입니다.

## 4. 행동 변수 만들기

**질문**: `active_months`(구매한 월의 수), 신선식품·할인 결제 비중(`fresh_ratio`, `discount_ratio`), 채널 변수(`multi_channel`, `main_channel`)는 어떻게 계산하는가?

In [ ]:
# 구매일에서 연월을 추출해 txn_ym 열로 추가합니다.
txns_before['txn_ym'] = txns_before['txn_date'].dt.strftime(date_format='%Y-%m')

# 고객별 구매한 월의 수, 신선식품 구매 금액 합계, 할인 금액 합계, 이용 채널 수를 계산하고 behavior에 할당합니다.
behavior = txns_before.groupby(by='customer_id').agg(active_months=('txn_ym', 'nunique'),
                                                     fresh_amount=('fresh_amount', 'sum'),
                                                     discount_amount=('discount_amount', 'sum'),
                                                     n_channels=('channel', 'nunique'))

# 구매 금액 대비 신선식품 금액의 비중을 fresh_ratio로 추가합니다.
# [참고] 거래별 비중의 평균이 아니라 금액 합계끼리 나눈 값이므로 금액이 큰 거래가 더 많이 반영됩니다.
behavior['fresh_ratio'] = (behavior['fresh_amount'] / rfm['monetary']).round(decimals=3)

# 구매 금액 대비 할인 금액의 비중을 discount_ratio로 추가합니다.
behavior['discount_ratio'] = (behavior['discount_amount'] / rfm['monetary']).round(decimals=3)

# 2개 이상 채널을 이용했으면 1, 아니면 0인 multi_channel을 추가합니다.
behavior['multi_channel'] = behavior['n_channels'].ge(other=2).astype(dtype=int)

# 고객별로 구매 횟수가 가장 많은 채널을 main_channel로 추가합니다.
# [참고] 횟수가 같은 채널이 여럿이면 거래 내역에서 먼저 나온(먼저 이용한) 채널을 선택합니다.
behavior['main_channel'] = txns_before.groupby(by='customer_id')['channel'].agg(func=lambda s: s.value_counts().index[0])

# behavior의 처음 5행을 확인합니다.
behavior.head()

**해석**
- `active_months`는 구매 횟수와 달리 **몇 달에 걸쳐** 샀는지를 나타냅니다.
  - 구매 횟수가 4회로 같더라도 한 달에 몰아서 산 고객과 4개월 동안 한 달에 한 번씩 나눠 산 고객은 브랜드 충성도가 다를 수 있습니다.
- 비중 변수는 금액 차이를 없애고 구매 성향만 비교할 수 있게 해 줍니다.

**질문**: 주 이용 채널이 동률인 고객은 몇 명이고, 이때 주 이용 채널은 어떻게 정하는가?

In [ ]:
# 고객·채널별 구매 횟수를 channel_counts에 할당합니다.
channel_counts = txns_before.groupby(by=['customer_id', 'channel']).size()

# 고객별로 구매 횟수가 가장 많은 채널이 둘 이상(동률)인지 여부를 channel_is_tie에 할당합니다.
channel_is_tie = channel_counts.groupby(level='customer_id').apply(func=lambda s: s.eq(other=s.max()).sum() > 1)

# 주 이용 채널 동률 고객 수를 확인합니다.
print('주 이용 채널 동률 고객 수:', channel_is_tie.sum())

**해석**
- 마트와 슈퍼를 2회씩 이용한 고객처럼 **동률**이 생각보다 많습니다.
- 주 이용 채널이 동률일 때 어느 채널을 주 채널로 볼지는 미리 정해둔 **업무 규칙**으로 판단합니다.
  - 먼저 이용한 채널, 합계 금액이 큰 채널, 최근 이용한 채널 등 명확한 기준을 정하고 문서로 남깁니다.
  - 업무 규칙이 명확하면 같은 분석을 반복해도 항상 같은 결과를 얻을 수 있습니다.

## 5. 목표변수 만들기

**질문**: 기준일 다음 달(M7)에 1회 이상 구매한 고객을 재구매 고객(`repurchase_m7 = 1`)으로 표시합니다.

In [ ]:
# 기준일 다음 달의 말일을 계산합니다.
# [참고] 기준일이 월말이므로 MonthEnd(1)을 더하면 다음 달 말일이 됩니다.
m7_end_date = txns_cutoff['cutoff_date'] + pd.offsets.MonthEnd(1)

# 기준일 다음 날부터 다음 달 말일까지의 거래인지 여부를 txn_in_m7에 할당합니다.
txn_in_m7 = after_cutoff & txns_cutoff['txn_date'].le(other=m7_end_date)

# M7 거래가 있는 고객 ID를 m7_buyers에 할당합니다.
m7_buyers = txns_cutoff.loc[txn_in_m7, 'customer_id'].unique()

# 분석 대상 고객의 재구매 여부를 target에 할당합니다.
target = active_customers['customer_id'].isin(values=m7_buyers).astype(dtype=int)

# 재구매율을 확인합니다.
print(f'재구매율: {target.mean():.3f}')

**해석**
- 목표변수 기간을 '기준일 이후 전체'가 아니라 **M7 한 달**로 정확히 잘라야 합니다.
  - 기준일 이후 전체 거래로 정하면, 첫 구매가 빠른 고객일수록 데이터 추출일까지 남은 기간이 길어 재구매할 시간이 더 많아집니다.
  - 고객마다 조건이 다르면 공정한 비교가 되지 않습니다.

## 6. 분석 테이블 완성과 검증

**질문**: 고객 마스터, RFM 변수, 행동 변수, 목표변수를 합쳐 분석 테이블을 만들면 `repurchase.csv`와 같은가?

In [ ]:
# 고객 마스터에서 분석에 쓸 열 이름을 리스트로 생성합니다.
master_cols = ['customer_id', 'cohort_ym', 'cutoff_date', 'age_group', 'app_member', 'distance_km']

# 고객 마스터에서 분석에 쓸 열을 선택하고 목표변수를 추가해 repurchase_made에 할당합니다.
repurchase_made = active_customers[master_cols].assign(repurchase_m7=target)

# RFM 변수와 행동 변수에서 사용할 열 이름을 리스트로 생성합니다.
feature_cols = ['frequency', 'monetary', 'active_months', 'recency_days', 'fresh_ratio', 'discount_ratio', 
                'main_channel', 'multi_channel']

# RFM 변수와 행동 변수를 고객 ID 기준으로 결합하고 features에 할당합니다.
features = rfm.join(other=behavior)[feature_cols]

# repurchase_made에 customer_id 기준으로 features를 결합합니다.
repurchase_made = repurchase_made.merge(right=features, how='left', on='customer_id')

# 기준일을 날짜시간형에서 문자열로 바꿉니다.
repurchase_made['cutoff_date'] = repurchase_made['cutoff_date'].dt.strftime(date_format='%Y-%m-%d')

# 열 순서를 repurchase와 같게 맞추고 인덱스를 초기화합니다.
repurchase_made = repurchase_made[repurchase.columns].reset_index(drop=True)

# repurchase_made의 처음 5행을 확인합니다.
repurchase_made.head()

In [ ]:
# 열별로 모든 값이 일치하는지 확인합니다.
repurchase_made.eq(other=repurchase).all()

**해석**
- 모든 열이 `True`이면, 거래 내역에서 직접 만든 테이블이 로지스틱 회귀 실습에서 사용한 분석 테이블과 같다는 뜻입니다.
- 실무에서는 이렇게 만든 테이블을 파일로 저장해 두고, 분석과 모형 학습에 반복해서 사용합니다.
- `how='left'`로 결합했으므로 기준일 이전 거래가 없는 고객이 있었다면 RFM 변수가 결측값으로 나타났을 것입니다.

## 7. RFM 점수와 고객 등급

**질문**: 최근성·구매 횟수·구매 금액을 각각 1 ~ 5점으로 바꾸면, 점수에 따라 재구매율이 어떻게 다른가? [RFM 분석]

In [ ]:
# 최근성 점수를 계산합니다. 최근에 구매할수록 높은 점수입니다.
# [참고] pd.qcut()은 값을 같은 인원이 들어가도록 5등분합니다.
# [참고] 같은 값이 많으면 구간 경계가 겹치므로 rank(method='first')로 순위로 바꾼 뒤 나눕니다.
# [참고] 이때 같은 값은 행 순서대로 다른 순위를 받으므로 값이 같은 고객도 점수가 다를 수 있습니다.
repurchase_made['R'] = pd.qcut(x=repurchase_made['recency_days'].rank(method='first'), q=5, labels=[5, 4, 3, 2, 1])

# 구매 횟수 점수를 계산합니다. 클수록 높은 점수입니다.
repurchase_made['F'] = pd.qcut(x=repurchase_made['frequency'].rank(method='first'), q=5, labels=[1, 2, 3, 4, 5])

# 구매 금액 점수를 계산합니다. 클수록 높은 점수입니다.
repurchase_made['M'] = pd.qcut(x=repurchase_made['monetary'].rank(method='first'), q=5, labels=[1, 2, 3, 4, 5])

# 세 점수를 더한 RFM 합계 점수를 추가합니다.
repurchase_made['RFM'] = repurchase_made[['R', 'F', 'M']].sum(axis=1)

# RFM 합계 점수별 고객 수와 재구매율을 계산하고 rfm_rate에 할당합니다.
rfm_rate = repurchase_made.groupby(by='RFM')['repurchase_m7'].agg(func=['count', 'mean'])

# rfm_rate를 소수점 3자리까지 반올림해서 확인합니다.
rfm_rate.round(decimals=3)

In [ ]:
# RFM 합계 점수별 재구매율을 막대 그래프로 시각화합니다.
plt.figure(figsize=(6, 4), dpi=100)
sns.barplot(x=rfm_rate.index, y=rfm_rate['mean'], color='silver')
for i, v in enumerate(rfm_rate['mean']):
    plt.text(x=i, y=v + 0.02, s=f'{v*100:.1f}%', ha='center', fontweight='bold')
plt.title(label='RFM 합계 점수별 7차월 재구매율')
plt.xlabel(xlabel='RFM 합계 점수 (3 ~ 15)')
plt.ylabel(ylabel='재구매율')
plt.ylim(bottom=0, top=1)
plt.show()

**해석**
- RFM 합계 점수가 높을수록 재구매율이 대체로 높아집니다.
  - 3점은 9.2%, 15점은 83.6%입니다.
  - 3점(9.2%)과 4점(8.1%)은 순서가 바뀌지만 차이가 작습니다.
- RFM 분석은 회귀 모형 없이도 고객을 등급으로 나눌 수 있어 현업에서 널리 씁니다.
  - 세 변수에 같은 가중치를 부여하므로, 각 변수가 재구매와 어떤 관련이 있는지는 로지스틱 회귀로 확인합니다.
- RFM 변수를 등분하는 구간과 점수 합산 방식은 정해진 규칙이 아니므로 업무 목적에 맞게 정합니다.

## 정리

- 입력변수는 기준일 이전 거래로만 만들고, 기준일 이후 거래는 목표변수를 만들 때만 사용했습니다.
- 거래 내역을 고객 단위로 집계해 RFM 변수(최근성·구매 횟수·구매 금액)와 행동 변수(활동 월 수, 비중, 채널)를 만들었습니다.
- 주 이용 채널처럼 동률이 생기는 변수는 처리 규칙을 정하고 문서로 남깁니다.
- 직접 만든 테이블이 로지스틱 회귀 실습의 분석 테이블과 같음을 확인했습니다.
- RFM 점수로 고객을 등급으로 나누면 회귀 모형 없이도 재구매 가능성이 높은 고객을 고를 수 있습니다.

## End of Document